# One-off private CPU cache diagnostic d01
Observe the unchanged, fingerprint-checked accepted verify_cache function on private mounted cache v5.
No model imports/loading/inference, training, BigQuery, network, GPU, model copying or benchmark identity reservation.
Per-file/byte/read-time/CPU/heartbeat observations are operational diagnostics, not scientific completion or benchmark latency.
Diagnostic runtime is default Kaggle CPU/Python, different from pinned T4 inference runtime.
1620s parent watchdog; 1800s platform cap. Keep every output; no retries or changes to pins/checks/caps.


In [ ]:
ALLOW_DIAGNOSTIC = True  # enabled only for this separately authorized diagnostic


In [ ]:
DRIVER_SHA256 = '6f72f78d6ae1d5b6d6a56d2f5b527c3bc08da202e6bc306bdb28372a375f192f'
DRIVER_SOURCE = '"""ONE-OFF CPU diagnostic, outside benchmark/bundle; never inference or a cache bypass."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport platform\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom datetime import UTC, datetime\nfrom pathlib import Path\n\nCAP_SECONDS = 1620\nSETUP_SHA = "91be7869ae7223b93e7ff6c3c7a58589b241161ef7a782351f02e105e86d9218"\nPINS_SHA = "6f72c03fe03c379b1ed0844192ef17c94cb60e908c5269f0e49750a32f04998a"\nINVENTORY_SHA = "ae3b7dc2361f151bcc59d68f4b2ddcdef74b99a5fe8374b041190bded84d84f1"\nBUNDLE_SHA = "2aa4dac0cb4110783279c3361b4c7618ce601402ed3a40cef168adad75caa6de"\n\n\ndef sha(path):\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef write_new(path, obj):\n    with path.open("x") as f:\n        json.dump(obj, f, sort_keys=True, indent=2)\n        f.write("\\n")\n\n\ndef load_native(script):\n    if sha(script) != SETUP_SHA:\n        raise ValueError("Accepted verifier fingerprint mismatch")\n    spec = importlib.util.spec_from_file_location("accepted_cache_verifier", script)\n    m = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(m)\n    return m\n\n\ndef observe(native, pins, home, output, *, heartbeat_seconds=10):\n    """Run exact accepted verify_cache while measuring its actual Path.open/read calls."""\n    started = time.monotonic()\n    original_open = Path.open\n    old_trace = sys.gettrace()\n    journal = output / "cache-progress.jsonl"\n    journal.touch(exist_ok=False)\n    lock = threading.RLock()\n    state = dict(phase="starting", total_bytes_read=0, files_read_complete=0)\n    stop = threading.Event()\n    source = Path(native.verify_cache.__code__.co_filename).read_text()\n    function = next(\n        n\n        for n in ast.parse(source).body\n        if isinstance(n, ast.FunctionDef) and n.name == "verify_cache"\n    )\n    assignment = next(\n        n\n        for n in ast.walk(function)\n        if isinstance(n, ast.Assign)\n        and any(isinstance(t, ast.Name) and t.id == "path" for t in n.targets)\n    )\n    native_code = native.verify_cache.__code__\n\n    def event(kind, **extra):\n        with lock:\n            now = time.monotonic()\n            snapshot = {k: v for k, v in state.items() if not k.startswith("_")}\n            if "_operation_started" in state:\n                snapshot["current_operation_elapsed_seconds"] = now - state["_operation_started"]\n            payload = dict(event=kind, elapsed_seconds=now - started, **snapshot, **extra)\n            with original_open(journal, "a") as stream:\n                stream.write(json.dumps(payload, sort_keys=True) + "\\n")\n                stream.flush()\n\n    def heartbeat():\n        while not stop.wait(heartbeat_seconds):\n            event("heartbeat")\n\n    def trace(frame, kind, arg):\n        if frame.f_code is not native_code:\n            return None\n        if kind == "line" and frame.f_lineno == assignment.lineno:\n            local = frame.f_locals\n            with lock:\n                state.update(\n                    file=local["kind"] + "/" + local["name"],\n                    expected_bytes=local["evidence"]["bytes"],\n                    phase="path_and_size_checks",\n                    file_bytes_read=0,\n                    read_wall_seconds=0.0,\n                    read_cpu_seconds=0.0,\n                    hash_and_loop_wall_seconds=0.0,\n                    _file_started=time.monotonic(),\n                    _operation_started=time.monotonic(),\n                )\n                state.pop("_previous_read_finished", None)\n                event("file_started")\n        return trace\n\n    class Reader:\n        def __init__(self, handle):\n            self.handle = handle\n\n        def __enter__(self):\n            self.handle.__enter__()\n            return self\n\n        def __exit__(self, *args):\n            result = self.handle.__exit__(*args)\n            with lock:\n                state["phase"] = "file_read_complete_hash_comparison_pending"\n                if args[0] is None and state["file_bytes_read"] == state["expected_bytes"]:\n                    state["files_read_complete"] += 1\n                event(\n                    "file_read_complete",\n                    file_wall_seconds=time.monotonic() - state["_file_started"],\n                )\n            return result\n\n        def read(self, size=-1):\n            now = time.monotonic()\n            with lock:\n                if "_previous_read_finished" in state:\n                    state["hash_and_loop_wall_seconds"] += now - state["_previous_read_finished"]\n                state.update(phase="read", _operation_started=now)\n            cpu = time.thread_time()\n            block = self.handle.read(\n                size\n            )  # unchanged native request size and actual returned bytes\n            cpu_elapsed = time.thread_time() - cpu\n            finished = time.monotonic()\n            with lock:\n                state["read_wall_seconds"] += finished - now\n                state["read_cpu_seconds"] += cpu_elapsed\n                state["file_bytes_read"] += len(block)\n                state["total_bytes_read"] += len(block)\n                state.update(\n                    phase="hash_and_loop",\n                    _previous_read_finished=finished,\n                    _operation_started=finished,\n                )\n            return block\n\n    def observed_open(path, *args, **kwargs):\n        # Only calls from the accepted verification function are measured.\n        if sys._getframe(1).f_code is not native_code:\n            return original_open(path, *args, **kwargs)\n        with lock:\n            state.update(phase="file_open", _operation_started=time.monotonic())\n            event("file_open_started")\n        handle = original_open(path, *args, **kwargs)\n        with lock:\n            event("file_open_finished")\n        return Reader(handle)\n\n    event(\n        "verification_started",\n        expected_files=sum(len(m["files"]) for m in pins["models"].values()),\n        expected_total_bytes=sum(\n            e["bytes"] for m in pins["models"].values() for e in m["files"].values()\n        ),\n    )\n    thread = threading.Thread(target=heartbeat, daemon=True)\n    thread.start()\n    try:\n        Path.open = observed_open\n        sys.settrace(trace)\n        receipt = native.verify_cache(pins, home, "all")\n        with lock:\n            state["phase"] = "all_native_hash_checks_passed"\n            event("verification_completed")\n        return receipt\n    except BaseException as exc:\n        event("verification_failed", error_type=type(exc).__name__, error=str(exc))\n        raise\n    finally:\n        sys.settrace(old_trace)\n        Path.open = original_open\n        stop.set()\n        thread.join(timeout=2)\n\n\ndef worker(mount, output):\n    archives = list(mount.rglob("input-bundle.bundle"))\n    inventories = list(mount.rglob("b12-cache-inventory.json"))\n    if len(archives) != 1 or len(inventories) != 1:\n        raise ValueError("Require one accepted setup dataset and one cache-v5 inventory")\n    archive, inventory = archives[0], inventories[0]\n    kit = archive.parent\n    script = kit / "scripts/26_kaggle_environment.py"\n    pins_path = kit / "configs/kaggle-b12-model-pins.json"\n    home = inventory.parent / "hf-home"\n    for path, expected in [\n        (archive, BUNDLE_SHA),\n        (inventory, INVENTORY_SHA),\n        (script, SETUP_SHA),\n        (pins_path, PINS_SHA),\n    ]:\n        if not path.resolve().is_relative_to(mount.resolve()) or sha(path) != expected:\n            raise ValueError("Diagnostic input fingerprint/path mismatch")\n    if not home.resolve().is_relative_to(mount.resolve()):\n        raise ValueError("Mounted cache escaped input boundary")\n    native = load_native(script)\n    pins = native.load_pins(pins_path)\n    write_new(\n        output / "diagnostic-provenance.json",\n        dict(\n            started_at_utc=datetime.now(UTC).isoformat(),\n            script_sha256=sha(script),\n            pins_sha256=sha(pins_path),\n            bundle_sha256=sha(archive),\n            inventory_sha256=sha(inventory),\n            python=sys.version,\n            platform=platform.platform(),\n            logical_cpus=os.cpu_count(),\n            comparison_limit="CPU host/default Python; differs from pinned T4 benchmark runtime",\n            model_loaded=False,\n            inference=False,\n            network_enabled=False,\n            weights_copied=False,\n            scientific_ready=False,\n            benchmark_attempt_reserved=False,\n        ),\n    )\n    receipt = observe(native, pins, home, output)\n    write_new(output / "native-cache-receipt.json", receipt)\n    print("Complete accepted cache hash verification; no inference.", flush=True)\n\n\ndef run_session(mount, output, source):\n    output.mkdir(parents=True, exist_ok=False)\n    started = time.monotonic()\n    write_new(\n        output / "diagnostic-settings.json",\n        dict(\n            diagnostic_only=True,\n            cpu_only=True,\n            verification_cap_seconds=CAP_SECONDS,\n            platform_cap_seconds=1800,\n            model_loaded=False,\n            inference=False,\n            benchmark_attempt_reserved=False,\n            scientific_ready=False,\n            accepted_setup_sha256=SETUP_SHA,\n        ),\n    )\n    script = output / "diagnostic-driver.py"\n    script.write_text(source)\n    summary = dict(\n        status="failed",\n        inference=False,\n        model_loaded=False,\n        scientific_ready=False,\n        benchmark_attempt_reserved=False,\n        diagnostic_only=True,\n        verification_cap_seconds=CAP_SECONDS,\n    )\n    process = None\n    try:\n        env = {\n            k: os.environ[k]\n            for k in ("PATH", "LD_LIBRARY_PATH", "LANG", "LC_ALL")\n            if k in os.environ\n        }\n        env.update(\n            PYTHONDONTWRITEBYTECODE="1",\n            HF_HUB_OFFLINE="1",\n            TRANSFORMERS_OFFLINE="1",\n            HF_HUB_DISABLE_IMPLICIT_TOKEN="1",\n        )\n        with (output / "worker.log").open("x") as log:\n            process = subprocess.Popen(\n                [\n                    sys.executable,\n                    str(script),\n                    "--worker",\n                    "--mount",\n                    str(mount),\n                    "--output",\n                    str(output),\n                ],\n                stdout=log,\n                stderr=subprocess.STDOUT,\n                env=env,\n                start_new_session=True,\n            )\n            code = process.wait(timeout=max(0.0, CAP_SECONDS - (time.monotonic() - started)))\n        summary["returncode"] = code\n        if code:\n            raise RuntimeError("Diagnostic worker failed; inspect retained worker.log")\n        summary["status"] = "completed_cache_diagnostic"\n    except BaseException as exc:\n        if process is not None:\n            try:\n                os.killpg(process.pid, signal.SIGKILL)\n            except ProcessLookupError:\n                pass\n            process.wait()\n        summary.update(\n            status="timed_out" if isinstance(exc, subprocess.TimeoutExpired) else "failed",\n            error_type=type(exc).__name__,\n        )\n        raise\n    finally:\n        summary.update(\n            wall_seconds=time.monotonic() - started,\n            finished_at_utc=datetime.now(UTC).isoformat(),\n        )\n        write_new(output / "diagnostic-summary.json", summary)\n        write_new(\n            output / "artifacts-manifest.json",\n            {\n                p.name: dict(bytes=p.stat().st_size, sha256=sha(p))\n                for p in output.iterdir()\n                if p.is_file()\n            },\n        )\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--worker", action="store_true")\n    parser.add_argument("--mount", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    if not args.worker:\n        parser.error("Only private notebook session orchestration or explicit diagnostic worker")\n    worker(args.mount, args.output)\n'


In [ ]:
if not ALLOW_DIAGNOSTIC:
    print("BLOCKED: diagnostic disabled")
else:
    import hashlib
    from pathlib import Path
    if hashlib.sha256(DRIVER_SOURCE.encode()).hexdigest() != DRIVER_SHA256:
        raise ValueError("Diagnostic source fingerprint mismatch")
    scope = {"__name__": "diagnostic_notebook"}
    exec(compile(DRIVER_SOURCE, "diagnostic-driver", "exec"), scope)
    scope["run_session"](Path("/kaggle/input"), Path("/kaggle/working/cache-diagnostic-d01"), DRIVER_SOURCE)
